In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import  Dataset,Dataset,DataLoader,random_split
from torchvision import transforms
from pycocotools.coco import COCO

The Dataloader load the data , by tokenizing the text, and returning image , and mask

In [ ]:
class segment_dataloader(Dataset):
  def __init__(self,image_dir,annot_dir,token,image_size = (512, 512)):
    self.image_dir = image_dir
    self.annot_dir = annot_dir
    self.token = token
    self.coco = COCO(self.annot_dir)
    self.annotations = list(self.coco.anns.values())
    self.categories = self.coco.loadCats(self.coco.getCatIds())
    self.category_names =  {
        cat['id']: cat['name']
        for cat in self.categories
    }
    self.resize_pad = ResizeAndPad(512)
  def __len__(self):
    return len(self.annotations)

  def __getitem__(self,idx):
    ann = self.annotations[idx]
    imageid = ann['image_id']
    imageinfo = self.coco.loadImgs(imageid)[0]
    img = os.path.join(self.image_dir,imageinfo['file_name'])
    image =Image.open(img).convert("RGB")
    mask = self.coco.annToMask(ann)
    mask = Image.fromarray(mask)
    image, mask = self.resize_pad(image, mask)
    category_name = self.category_names[
            ann["category_id"]
        ]
    category_tok = self.token.encode(category_name)
    image = TF.to_tensor(image)
    mask = TF.pil_to_tensor(mask)
    return image,category_tok,mask



In [ ]:
class image_encoder(nn.Module):
  def __init__(self,input_channel):
    super().__init__()
    self.input_channel = input_channel
    self.image_enc = nn.Sequential(
        nn.Conv2d(input_channel , 32,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(32,64,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(64,64,3,padding=1),
        nn.ReLU()

    )
  def forward(self,X):
    out = self.image_enc(X)
    return out

class CrossAttention(nn.Module):
  def __init__(self,d_model,image_d,embed_dim):
    super().__init__()
    self.w_k= nn.Linear(image_d,embed_dim,bias =False)
    self.w_q = nn.Linear(d_model,embed_dim,bias =False)
    self.w_v = nn.Linear(image_d,embed_dim,bias =False)
  def forward(self,image_enc,text_emb):
    K = self.w_k(image_enc)
    Q = self.w_q(text_emb)
    V =self.w_v(image_emb)
    A = torch.softmax(Q @ K.T,dim=-1)
    Z = A @ V
    return Z, A

In [ ]:
text_emb= torch.randn(1,13)
image_emb = torch.randn(1025,1025)
attn = CrossAttention(13,1025,13)
out,c = attn(image_emb,text_emb)